<a href="https://colab.research.google.com/github/LibanioMatheus/Aurora-fase3/blob/main/00_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema Inteligente — Colônia Aurora Siger

## Notebook Principal (Capítulo 1)

Este notebook é **autossuficiente**: contém todo o código dos
três módulos integrados, sem depender de imports externos.
Basta executar as células na ordem.

Integra:

- **Organização dos dados** — estruturas e hierarquia;
- **Lógica de decisão** — regras e análise de energia;
- **Previsão** — regressão linear simples.

Fluxo geral: **entrada → processamento → saída**.

## Parte 1 — Organização dos dados

In [ ]:
def criar_colonia():
    """Monta a estrutura hierarquica da colonia."""
    colonia = {
        "energetico": {
            "solar": {
                "geracao": 45,
                "historico_geracao": [30, 34, 38, 41, 45],
            },
            "eolico": {
                "geracao": 25,
                "historico_vento": [8, 10, 12, 13, 15],
                "historico_geracao": [18, 21, 25, 27, 31],
            },
            "reserva_baterias": 60,
        },
        "ambiental": {
            "temperatura_interna": 22,
            "temperatura_externa": -55,
            "velocidade_vento": 14,
            "previsao_tempestade": False,
        },
        "operacional": {
            "consumo_total": 70,
            "modulos": {
                "suporte_vida": {"consumo": 30, "essencial": True},
                "estufa": {"consumo": 20, "essencial": False},
                "laboratorio": {"consumo": 12, "essencial": False},
                "comunicacao": {"consumo": 8, "essencial": True},
            },
        },
    }
    return colonia


def acessar(colonia, caminho):
    """Navega pela hierarquia usando uma lista de chaves."""
    atual = colonia
    for chave in caminho:
        if isinstance(atual, dict) and chave in atual:
            atual = atual[chave]
        else:
            return None
    return atual


def energia_total_gerada(colonia):
    """Soma a geracao solar e eolica."""
    solar = acessar(colonia, ["energetico", "solar", "geracao"])
    eolico = acessar(colonia, ["energetico", "eolico", "geracao"])
    return solar + eolico


def resumo(colonia):
    """Imprime um resumo legivel do estado da colonia."""
    print("=" * 50)
    print("RESUMO DO ESTADO DA COLONIA AURORA SIGER")
    print("=" * 50)
    print("Geracao solar .......:", acessar(colonia, ["energetico","solar","geracao"]))
    print("Geracao eolica ......:", acessar(colonia, ["energetico","eolico","geracao"]))
    print("Geracao total .......:", energia_total_gerada(colonia))
    print("Reserva (baterias) ..:", acessar(colonia, ["energetico","reserva_baterias"]))
    print("Consumo total .......:", acessar(colonia, ["operacional","consumo_total"]))
    print("Temp. interna .......:", acessar(colonia, ["ambiental","temperatura_interna"]), "C")
    print("Velocidade do vento .:", acessar(colonia, ["ambiental","velocidade_vento"]))
    print("Tempestade prevista .:", acessar(colonia, ["ambiental","previsao_tempestade"]))
    print("=" * 50)

## Parte 2 — Previsão (regressão linear simples)

In [ ]:
def ajustar_reta(x, y):
    """Calcula os coeficientes da reta y = a*x + b (minimos quadrados)."""
    n = len(x)
    if n == 0 or n != len(y):
        raise ValueError("Listas x e y devem ter o mesmo tamanho.")
    soma_x = sum(x)
    soma_y = sum(y)
    soma_xy = sum(x[i] * y[i] for i in range(n))
    soma_x2 = sum(x[i] * x[i] for i in range(n))
    denominador = n * soma_x2 - soma_x * soma_x
    if denominador == 0:
        return 0.0, soma_y / n
    a = (n * soma_xy - soma_x * soma_y) / denominador
    b = (soma_y - a * soma_x) / n
    return a, b


def prever(a, b, x_novo):
    """Estima y a partir de um novo x."""
    return a * x_novo + b


def r_quadrado(x, y, a, b):
    """Calcula o coeficiente de determinacao R^2."""
    n = len(y)
    media_y = sum(y) / n
    soma_erro = sum((y[i] - (a * x[i] + b)) ** 2 for i in range(n))
    soma_total = sum((y[i] - media_y) ** 2 for i in range(n))
    if soma_total == 0:
        return 1.0
    return 1 - (soma_erro / soma_total)


def prever_energia_eolica(historico_vento, historico_geracao, vento_previsto):
    """Preve a energia eolica futura a partir dos historicos."""
    a, b = ajustar_reta(historico_vento, historico_geracao)
    estimativa = prever(a, b, vento_previsto)
    qualidade = r_quadrado(historico_vento, historico_geracao, a, b)
    return {
        "vento_previsto": vento_previsto,
        "energia_estimada": round(estimativa, 1),
        "coef_angular": round(a, 3),
        "coef_linear": round(b, 3),
        "r2": round(qualidade, 3),
    }

## Parte 3 — Lógica de decisão e análise de energia

In [ ]:
def analisar_energia(geracao, consumo, reserva=0):
    """Compara geracao, consumo e reserva e devolve uma acao simples."""
    saldo = geracao - consumo
    if consumo > geracao:
        situacao = "RISCO"
        mensagem = "ALERTA: consumo maior que geracao"
    elif geracao > consumo:
        situacao = "SOBRA"
        mensagem = "SUGESTAO: armazenar energia excedente"
    else:
        situacao = "EQUILIBRIO"
        mensagem = "OK: geracao e consumo equilibrados"
    return {
        "geracao": geracao, "consumo": consumo, "reserva": reserva,
        "saldo": saldo, "situacao": situacao, "mensagem": mensagem,
    }


def decidir_acao(energia, consumo, previsao_tempestade=False):
    """Toma a decisao operacional principal da colonia."""
    consumo_alto = consumo >= 60
    energia_critica = energia < 30
    energia_baixa = energia < 50

    if energia_critica and consumo_alto:
        nivel, acao = "CRITICO", "ATIVAR MODO DE EMERGENCIA"
        detalhe = "Energia critica com consumo alto."
    elif previsao_tempestade and energia_baixa:
        nivel, acao = "ALTO", "ATIVAR MODO DE ECONOMIA"
        detalhe = "Tempestade de areia prevista e energia baixa."
    elif energia_baixa:
        nivel, acao = "MEDIO", "REDUZIR CONSUMO"
        detalhe = "Energia abaixo do nivel seguro (50)."
    else:
        nivel, acao = "NORMAL", "MANTER SISTEMAS NORMAIS"
        detalhe = "Condicoes dentro do esperado."

    return {
        "entrada": {"energia": energia, "consumo": consumo,
                    "tempestade": previsao_tempestade},
        "nivel": nivel, "acao": acao, "detalhe": detalhe,
    }


def priorizar_modulos(modulos, modo_economia):
    """Define quais modulos ficam LIGADOS e quais sao DESLIGADOS."""
    ligados, desligados, consumo_final = [], [], 0
    for nome in modulos:
        info = modulos[nome]
        if info["essencial"]:
            ligados.append(nome)
            consumo_final += info["consumo"]
        elif modo_economia:
            desligados.append(nome)
        else:
            ligados.append(nome)
            consumo_final += info["consumo"]
    return {"ligados": ligados, "desligados": desligados,
            "consumo_final": consumo_final}

## Parte 4 — Ciclo integrado de análise e decisão

Executa um ciclo completo: lê os dados, analisa a energia, faz a
previsão, decide a ação e prioriza os módulos.

In [ ]:
def executar_ciclo(colonia):
    """Executa um ciclo completo de analise e decisao da colonia."""
    # 1. ENTRADA: leitura dos dados
    resumo(colonia)
    geracao = energia_total_gerada(colonia)
    consumo = acessar(colonia, ["operacional", "consumo_total"])
    reserva = acessar(colonia, ["energetico", "reserva_baterias"])
    tempestade = acessar(colonia, ["ambiental", "previsao_tempestade"])

    # 2. ANALISE DE ENERGIA
    print()
    print("[1] ANALISE DO USO DE ENERGIA")
    analise = analisar_energia(geracao, consumo, reserva)
    print("    Geracao=%s  Consumo=%s  Saldo=%s" %
          (analise["geracao"], analise["consumo"], analise["saldo"]))
    print("    ->", analise["mensagem"])

    # 3. PREVISAO (regressao linear)
    print()
    print("[2] PREVISAO DE ENERGIA EOLICA (regressao linear simples)")
    hist_vento = acessar(colonia, ["energetico", "eolico", "historico_vento"])
    hist_ger = acessar(colonia, ["energetico", "eolico", "historico_geracao"])
    vento_previsto = 11
    prev = prever_energia_eolica(hist_vento, hist_ger, vento_previsto)
    print("    Reta: energia = %s * vento + %s" %
          (prev["coef_angular"], prev["coef_linear"]))
    print("    Qualidade (R2) =", prev["r2"])
    print("    -> Para vento = %s, energia estimada ~ %s" %
          (prev["vento_previsto"], prev["energia_estimada"]))

    # 4. DECISAO OPERACIONAL
    print()
    print("[3] DECISAO OPERACIONAL (logica do sistema)")
    decisao = decidir_acao(geracao, consumo, tempestade)
    print("    Entrada:", decisao["entrada"])
    print("    Nivel:", decisao["nivel"])
    print("    ->", decisao["acao"], "(" + decisao["detalhe"] + ")")

    # 5. PRIORIZACAO DE MODULOS
    print()
    print("[4] PRIORIZACAO DE MODULOS")
    modo_economia = decisao["nivel"] in ("CRITICO", "ALTO", "MEDIO")
    modulos = acessar(colonia, ["operacional", "modulos"])
    prioridade = priorizar_modulos(modulos, modo_economia)
    print("    Modo economia ativo:", modo_economia)
    print("    Modulos LIGADOS ....:", prioridade["ligados"])
    print("    Modulos DESLIGADOS .:", prioridade["desligados"])
    print("    Consumo apos ajuste :", prioridade["consumo_final"])

    # 6. SAIDA FINAL
    print()
    print("=" * 50)
    print("DECISAO FINAL DO SISTEMA")
    print("=" * 50)
    print("  Situacao energetica :", analise["situacao"])
    print("  Acao recomendada    :", decisao["acao"])
    print("  Previsao eolica     : ~%s (vento %s)" %
          (prev["energia_estimada"], vento_previsto))
    print("=" * 50)

## Parte 5 — Execução do sistema

Execute a célula abaixo para rodar o ciclo completo da colônia.

In [ ]:
print("#" * 50)
print("#  SISTEMA INTELIGENTE - COLONIA AURORA SIGER  #")
print("#" * 50)
print()

colonia = criar_colonia()
executar_ciclo(colonia)

## Parte 6 — Demonstração de cenários

Mostra como a lógica de decisão responde a situações diferentes.

In [ ]:
print("########## DEMONSTRACAO DE CENARIOS ##########")
print()

cenarios = [
    {"nome": "Energia critica + consumo alto",
     "energia": 25, "consumo": 70, "tempestade": False},
    {"nome": "Tempestade prevista + energia baixa",
     "energia": 45, "consumo": 40, "tempestade": True},
    {"nome": "Energia baixa, sem tempestade",
     "energia": 48, "consumo": 35, "tempestade": False},
    {"nome": "Condicoes normais",
     "energia": 90, "consumo": 50, "tempestade": False},
]

for c in cenarios:
    d = decidir_acao(c["energia"], c["consumo"], c["tempestade"])
    print("-", c["nome"])
    print("  Entrada: energia=%s, consumo=%s, tempestade=%s" %
          (c["energia"], c["consumo"], c["tempestade"]))
    print("  Saida  : [%s] %s" % (d["nivel"], d["acao"]))
    print()